# 06 — Risk Parity / Equal Risk Contribution (ERC)

Notebook 05 found that the Most Diversified Portfolio (MDP) — a purely
decorrelation-driven objective — keeps the UUP corner that survives
covariance shrinkage in notebook 04. This notebook asks the same question
of a third risk-based objective: Equal Risk Contribution (ERC), which
allocates so that every asset contributes the same *share* of total
portfolio risk, $w_i(\Sigma w)_i = 1/N$ for all $i$.

ERC equalizes risk shares *by construction* — that part is an identity,
not a finding, and this notebook says so plainly rather than reporting it
as a result. The actual question is what an equal-risk-share allocation
does to the CASH weight UUP receives, and to concentration (effective N),
relative to GMV and MDP on the same $\Sigma$.

## §1 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt

import maplab as ml
import _nb06_helpers as h
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    EqualRiskContribution, risk_contributions, backtest, summary, mdp_closed_form,
)
from maplab.models import _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb06"))

## §3 Mechanism

- **ERC condition.** $w_i(\Sigma w)_i$ equal for all $i$, $\mathbf 1'w=1$,
  $w>0$. Solved via the strictly convex surrogate
  $\min_{y>0} \tfrac12 y'\Sigma y - \tfrac1N\sum_i\log y_i$, $w=y/\mathbf1'y$,
  whose stationarity condition $y_i(\Sigma y)_i = 1/N$ gives equal risk
  contributions exactly. The log barrier makes $w>0$ automatic — unlike the
  tangency/MDP QPs there is no boundary case and no fallback. The solution
  is unique.
- **Euler's theorem.** $\sum_i RC_i = \sigma_p$ exactly, where
  $RC_i = w_i(\Sigma w)_i/\sigma_p$ — risk is homogeneous of degree 1 in
  $w$, so it decomposes additively across assets with no residual.
- **Identity A** (this is the one that explains the cash weight):
  $w_i\cdot\beta_{i,p} = 1/N$ where $\beta_{i,p} = (\Sigma w)_i / w'\Sigma w$
  is asset $i$'s beta to the portfolio itself. An asset that *hedges* the
  portfolio has small (or negative) $\beta_{i,p}$, so buying its mandated
  $1/N$ share of total risk requires a correspondingly LARGE cash weight.
- **Identity B.** Risk weights $x = w\circ\sigma/(w'\sigma)$ equal ERC run
  on the correlation matrix $C=D^{-1}\Sigma D^{-1}$ — the same shape as
  notebook 05's MDP-on-$C$ result (there it was long-only GMV on $C$). Both
  are scale-invariant: $w(K\Sigma K)$ is proportional to $K^{-1}w(\Sigma)$
  for any diagonal $K>0$.
- **Special cases.** Constant correlation &#8658; inverse vol (any $N$);
  $N=2$ &#8658; inverse vol for any $\rho$.
- **Contrast with long-only GMV.** Every asset GMV actually *holds*
  (i.e. $w_i>0$) has $\beta_{i,p}=1$ exactly — that is GMV's own
  first-order condition restricted to its support. So GMV equalizes
  MARGINAL risk across the assets it holds, while ERC equalizes TOTAL risk
  contribution across ALL assets. This is verified numerically just below,
  not just asserted.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

snap_table, uup_context_table, gmv_beta_dev = h.snapshot_tables(panel, DATES)
print(snap_table.round(4))

print("\nUUP context (sample cov): mean pairwise correlation, vol rank (1 = lowest), "
      "mean off-diagonal correlation")
print(uup_context_table.round(4))

print("\nGMV first-order check: max|beta_i,p - 1| over assets with w_i > 1e-8 "
      "(should be ~0; the ~1e-5 floor is SLSQP's weight precision, not a failed identity)")
for d, v in gmv_beta_dev.items():
    print(f"  {d}: {v:.3e}")

### 3.3 &#8212; Figure F1: snapshot map at 2022-12-31

Where do GMV, MDP, ERC, MaxSharpe and EW sit in (vol, &mu;) space at the
train-window snapshot used to fix H1/H2's directions? &mu; here is a
252-day trailing sample mean, so the y-axis is illustrative &#8212; the
honest reading is the x-position (risk) and the printed weights, not
"expected return". MaxSharpe is computed at this single date only (a
snapshot reference, not backtested &#8212; the backtested runs are in
&#167;4).

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
n = Sigma_np.shape[0]
tickers = list(Sigma_2022.columns)

w_gmv_2022 = _min_variance_long_only(Sigma_2022)
w_mdp_2022, _ = _mdp_long_only(Sigma_2022, "F1", asof_2022)
w_erc_2022, _ = _erc_long_only(Sigma_2022, "F1", asof_2022)
rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "F1", asof_2022)
w_ew_2022 = np.full(n, 1.0 / n)

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"GMV": w_gmv_2022, "MDP": w_mdp_2022, "ERC": w_erc_2022,
     "MaxSharpe": w_msr_2022, "EqualWeight": w_ew_2022},
    title=f"Snapshot {asof_2022.date()} -- GMV / MDP / ERC / MaxSharpe / EW",
    figsize=(8, 6), legend_outside=False,
)
plt.show()

iU2022 = tickers.index("UUP")
print(f"w_UUP: GMV={w_gmv_2022[iU2022]:.4f}  MDP={w_mdp_2022[iU2022]:.4f}  ERC={w_erc_2022[iU2022]:.4f}  "
      f"MaxSharpe={w_msr_2022[iU2022]:.4f}  EW={w_ew_2022[iU2022]:.4f}")

### 3.4 &#8212; Figure F2: ERC mechanism grid

Cash weight vs. risk-contribution share, asset by asset, for GMV / MDP /
ERC / EW at the three snapshot dates &#8212; the picture behind Identity A.
Every ERC panel's risk-share bars sit exactly on the dashed 1/13 line by
construction (that is the *definition* of ERC, not a result); the
interesting bars are GMV's and MDP's, and the interesting gap in every
panel is between the risk-share bar and the cash-weight bar for UUP
specifically.

In [ ]:
fig, grid_table = h.mechanism_grid(panel, DATES)
plt.show()

print(grid_table.round(4))

In [ ]:
identA_dev = h.identity_a_maxdev(panel, asof_2022)

print(f"max|w_i - 1/(N*beta_i,p)| at {asof_2022.date()}: {identA_dev:.3e}")

## &#167;4 Backtests

Six fresh instances, same cost/rebalance/warm-up harness as notebooks
01&#8211;05. `ERC(S)` is the registered run; `GMV(S)`, `EW`, `MDP(S)` are
the reproduction-gate comparators (already backtested in earlier
notebooks); `ERC(LW)` is the covariance-estimator sensitivity; `IV` is the
notebook-local inverse-vol reference (never backtested with a shrinkage
estimator, same convention as notebook 05).

In [ ]:
strategies = {
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(LW)": EqualRiskContribution(cov_estimator=ml.ledoit_wolf_cov),
    "IV": ml.InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

GMV(S), EW, and MDP(S) must match notebooks 01/05 in full/train/test
Sharpe to 2 dp: GMV 0.62/0.59/0.76, EW 0.74/0.70/0.91, MDP(S)
0.76/0.70/0.98. If this doesn't hold, the harness or estimator wiring
changed something and everything downstream is suspect &#8212; so this
stops the notebook (`assert`) rather than just printing a warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
summary_table.round(4)

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "EW": "EW", "MDP(S)": "MDP_sample"})

print("REPRODUCTION GATE: PASSED -- GMV(S), EW, MDP(S) match notebooks 01/05 to 2 dp.")

## &#167;5 Mechanism checks, all 208 rebalances &#215; {sample, Ledoit-Wolf}

Recomputed fresh (not read off the &#167;4 backtests), so this is an
independent check of the solver itself, not a tautology. Tolerance 1e-11
unless noted &#8212; coordinate descent (the solver actually shipped in
`_erc_long_only`) delivers max deviations around 1e-12 per the read-only
design check that ran before this notebook was built, which is why the
tolerance is set at 1e-11.

In [ ]:
mech5_table = h.mechanism_checks(panel, rdates)
print(mech5_table.to_string(float_format=lambda v: f"{v:.3e}"))

for _, row in mech5_table.iterrows():
    assert row["max_rc_share_dev"] <= 1e-11, f"RC-share deviation failed: {row['max_rc_share_dev']}"
    assert row["max_euler"] <= 1e-12, f"Euler check failed: {row['max_euler']}"
    assert row["max_identA"] <= 1e-11, f"Identity A failed: {row['max_identA']}"
    assert row["max_identB"] <= 1e-11, f"Identity B failed: {row['max_identB']}"
    assert row["max_rho_bar_dev"] <= 1e-11, f"constant-correlation check failed: {row['max_rho_bar_dev']}"
    assert row["min_vol_margin_gmv"] >= -1e-11, f"vol ordering (GMV<=ERC) failed: {row['min_vol_margin_gmv']}"
    assert row["min_vol_margin_ew"] >= -1e-11, f"vol ordering (ERC<=EW) failed: {row['min_vol_margin_ew']}"
    assert row["min_w"] > 0, f"positivity failed: {row['min_w']}"

print("Mechanism checks: PASSED at all 208 rebalances, both estimators.")

n_sweeps_s = results["ERC(S)"]["strat"].sweeps
n_sweeps_lw = results["ERC(LW)"]["strat"].sweeps
print(f"strat.sweeps: ERC(S) median={np.median(n_sweeps_s):.0f} max={max(n_sweeps_s)}   "
      f"ERC(LW) median={np.median(n_sweeps_lw):.0f} max={max(n_sweeps_lw)}")

## §6 Hypothesis tests

### H1 — the UUP corner

In [ ]:
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_erc_ts < w_uup_gmv_ts
lt_mdp = w_uup_erc_ts < w_uup_mdp_ts
h1_full = lt_gmv & lt_mdp
n_full = len(h1_full)
test_mask = h1_full.index > split_ts
h1_test = h1_full.loc[test_mask]
n_test = int(test_mask.sum())

asof_2022 = pd.Timestamp("2022-12-31")
uup_2022 = {
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
}

_, Sigma_2022c = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np_2022c = Sigma_2022c.to_numpy()
w_erc_2022_arr = results["ERC(S)"]["wlog"].loc[asof_2022].reindex(Sigma_2022c.columns).to_numpy()
var_2022c = float(w_erc_2022_arr @ Sigma_np_2022c @ w_erc_2022_arr)
beta_p_2022c = (Sigma_np_2022c @ w_erc_2022_arr) / var_2022c
iU2022c = list(Sigma_2022c.columns).index("UUP")

print(f"H1 (ERC<GMV and ERC<MDP): full {int(h1_full.sum())}/{n_full} "
      f"({100*h1_full.sum()/n_full:.1f}%), test {int(h1_test.sum())}/{n_test} "
      f"({100*h1_test.sum()/n_test:.1f}%)")
print(f"2022-12-31 w_UUP: ERC={uup_2022['ERC(S)']:.4f}  GMV={uup_2022['GMV(S)']:.4f}  MDP={uup_2022['MDP(S)']:.4f}")
print(f"2022-12-31 beta_UUP,p (ERC) = {beta_p_2022c[iU2022c]:.4f}")
print(f"holds at 2022-12-31: ERC<GMV={uup_2022['ERC(S)'] < uup_2022['GMV(S)']}  "
      f"ERC<MDP={uup_2022['ERC(S)'] < uup_2022['MDP(S)']}")

### H2 — effective N

In [ ]:
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

comparator_max = pd.concat([effN_gmv_ts, effN_mdp_ts], axis=1).max(axis=1)
h2_full = effN_erc_ts >= comparator_max
n_full2 = len(h2_full)
test_mask2 = h2_full.index > split_ts
h2_test = h2_full.loc[test_mask2]
n_test2 = int(test_mask2.sum())

print(f"H2 (effN(ERC) >= max(GMV,MDP)): full {int(h2_full.sum())}/{n_full2} "
      f"({100*h2_full.sum()/n_full2:.1f}%), test {int(h2_test.sum())}/{n_test2} "
      f"({100*h2_test.sum()/n_test2:.1f}%)")
print(f"median effN: ERC(S)={float(effN_erc_ts.median()):.4f}  "
      f"GMV(S)={float(effN_gmv_ts.median()):.4f}  MDP(S)={float(effN_mdp_ts.median()):.4f}")
print(f"2022-12-31 effN: ERC={float(effN_erc_ts.loc[asof_2022]):.4f}  "
      f"GMV={float(effN_gmv_ts.loc[asof_2022]):.4f}  MDP={float(effN_mdp_ts.loc[asof_2022]):.4f}")

### H3 / H4 / H5 — paired tests, &#946; decomposition, cost drag

Pairs: ERC(S) &#8722; GMV(S), ERC(S) &#8722; MDP(S), ERC(S) &#8722; EW,
ERC(LW) &#8722; ERC(S), ERC(S) &#8722; IV. Plain OLS (no HAC &#8212; same
caveat as notebooks 03&#8211;05), realized daily simple excess return on
the market's simple excess return; the identity check is 0 by OLS
construction (mean residual is exactly 0 for an OLS fit with an
intercept), not a residual to be interpreted.

In [ ]:
pairs = [
    ("ERC(S)", "GMV(S)"),
    ("ERC(S)", "MDP(S)"),
    ("ERC(S)", "EW"),
    ("ERC(LW)", "ERC(S)"),
    ("ERC(S)", "IV"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
print(paired_table.round(4))

In [ ]:
print("cost drag %/yr per run (full window):")
for name in results:
    print(f"  {name:12s} {float(summary_table.loc[(name, 'full'), 'cost_drag']) * 100:.4f}%")

Single-strategy CAPM &#945;/&#946; below is **not pre-registered**
&#8212; shown for context only, no hypothesis is attached to any single
number in it, and it is Phase 2 multiple-testing material.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["ERC(S)", "GMV(S)", "MDP(S)", "EW", "ERC(LW)", "IV"],
                                     MKT_simple, rf_daily)
print(capm_table.round(4))

In [ ]:
# Direct regression of each pair's diff series on MKT: alpha_diff has its
# own t-stat (unlike alpha_a - alpha_b, which ignores the covariance
# between the two separate alpha estimates) -- this is what H3/H4/H5's
# "Delta-alpha |t| < 2" conditions are evaluated against.
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
print(diff_capm_table.round(4))

In [ ]:
decomp = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                         results, rf_daily, MKT_simple)
print(ml.inference.format_beta_decomposition(decomp, capm_table, diff_capm_table), end="")

## §7 Figures

### F3 — group-weight stackplot

Which asset groups carry the target weight over time under ERC(S) vs.
MDP(S) vs. GMV(S)?

In [ ]:
fig = ml.plotting.group_stackplot(results, ["ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts)
plt.show()

erc_group = ml.diagnostics.weights_by_group(results["ERC(S)"]["wlog"])
mdp_group = ml.diagnostics.weights_by_group(results["MDP(S)"]["wlog"])
gmv_group = ml.diagnostics.weights_by_group(results["GMV(S)"]["wlog"])

print("Mean target weight by asset group:")
group_means = pd.DataFrame({"ERC(S)": erc_group.mean(), "MDP(S)": mdp_group.mean(), "GMV(S)": gmv_group.mean()})
print(group_means.round(4))

### F4 — effN, max weight, and UUP weight over time

Is the effN/UUP contrast between ERC, MDP and GMV a snapshot artifact or
does it hold across the full 208-rebalance history?

Bottom row: trailing 12-rebalance one-way turnover (%/yr; initial allocation excluded), EW dashed.

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts,
    rows=("effN", "max_w", "highlight", "turnover"), figsize=(9.5, 14),
    suptitle="ERC(S) vs MDP(S) vs GMV(S): effN, max weight, UUP weight, turnover",
)
plt.show()

### F5 — cumulative paired difference

For each registered pair, is the daily return difference a persistent
drift or a noisy path with no clear direction?

In [ ]:
fig = ml.plotting.cumulative_paired(
    paired_series, pairs, split_ts,
    title="Cumulative paired daily return differences",
    ylabel="cumulative daily return difference (%)",
)
plt.show()

## Verdict table (registered rules applied mechanically)

Each registered rule from §2 is applied mechanically to the statistics printed above. The rule column shows the registered wording; `[applied: …]` states how an ambiguous rule was read. This table is an addition in Stage B; no statistic changed.

In [ ]:
reg = ml.inference.load_registration("nb06")

h1_holds_2022 = (uup_2022["ERC(S)"] < uup_2022["GMV(S)"]) and (uup_2022["ERC(S)"] < uup_2022["MDP(S)"])
d4 = decomp["ERC(S) - EW"]
t_full_ew = float(paired_table.loc[("ERC(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("ERC(S) - EW", "test (> split)"), "ann_mean_diff"])
t_full_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "test (> split)"), "t"])

stats = {
    "H1": dict(holds_2022=h1_holds_2022, frac_full=h1_full.sum() / n_full, frac_test=h1_test.sum() / n_test,
               display=f"full {int(h1_full.sum())}/{n_full}, test {int(h1_test.sum())}/{n_test}, "
                       f"2022 holds={h1_holds_2022}"),
    "H2": dict(frac_full=h2_full.sum() / n_full2, frac_test=h2_test.sum() / n_test2,
               display=f"full {int(h2_full.sum())}/{n_full2}, test {int(h2_test.sum())}/{n_test2}"),
}
for hid, pair_key in [("H3_GMV", "ERC(S) - GMV(S)"), ("H3_MDP", "ERC(S) - MDP(S)"), ("H5_IV", "ERC(S) - IV")]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    stats[hid] = dict(t=t, display=f"t={t:.2f}")
stats["H4"] = dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d4["beta_part"], d_alpha=d4["d_alpha"],
                   display=f"t_full={t_full_ew:.2f}, mean_test={mean_test_ew:.4f}, "
                           f"|beta_part|={abs(d4['beta_part']):.4f} vs |d_alpha|={abs(d4['d_alpha']):.4f}")
stats["H5_LW"] = dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}")

vt = ml.inference.evaluate(reg, stats)
print(vt.to_string(index=False))

## §8 TAKEAWAY

- **H1 — supported:** the UUP corner **shrinks** under ERC but does not
  disappear. w_UUP(ERC) &lt; w_UUP(GMV) and &lt; w_UUP(MDP) at 2022-12-31
  (48.36% vs 58.12% / 59.22%) and on 197/208 (94.7%) rebalances overall
  (test 33/40, 82.5%). At that same date &beta;<sub>UUP,p</sub>(ERC) =
  0.1591 &#8212; UUP is a weak hedge of the ERC portfolio itself, and by
  Identity A a low &beta;<sub>i,p</sub> is exactly what forces a large cash
  weight to still buy its mandated 1/13 share of risk. UUP's mean pairwise
  correlation with the rest of the universe at 2022-12-31 is &#8722;0.4659
  (it is also the single lowest-vol asset, rank 1/13, printed in &#167;3) &#8212;
  the same correlation-driven corner notebook 05 found for MDP. In
  **absolute** terms 48.36% in one asset is still a corner: ERC does not
  spread the portfolio evenly across the universe, it spreads *risk*
  evenly, and UUP's low correlation to everything else means a small
  amount of its risk buys a large amount of its notional.
- **The equal-risk-share result is an identity, not a finding.** Every
  ERC portfolio in this notebook has RC share = 1/13 for all 13 assets by
  *construction* &#8212; &#167;5 confirms this holds to &#8804; 1e-11 across
  all 208 rebalances &times; both estimators because the solver enforces
  it, not because of anything discovered about this universe. The actual
  empirical content of this notebook is what ERC does to *cash* weights and
  concentration, not that risk shares come out equal.
- **H2 — supported, decisively:** effN(ERC) &#8805; max(effN GMV, effN MDP)
  on 208/208 (100%) rebalances, test 40/40 (100%); median effN
  ERC(S) = 6.86 vs GMV(S) = 3.79, MDP(S) = 3.35. But even with risk shares
  *exactly* equal, effN(ERC) at 2022-12-31 is 3.88 &#8212; above GMV's 2.51
  and MDP's 2.47, but nowhere near EW's 13.00. Equalizing risk
  contribution is not the same thing as equalizing capital, and it is not a
  general cure for concentration when one asset (UUP) is both very
  low-vol and strongly anti-correlated with the rest of the book.
- **H3 — null holds (on the registered statistic):** &Delta;&alpha; from the
  direct diff-regression &#8212; the pre-registered H3 statistic &#8212; is
  +0.09%/yr for ERC(S) &#8722; GMV(S) (t = 0.29) and &#8722;0.26%/yr for
  ERC(S) &#8722; MDP(S) (t = &#8722;0.74); both &#124;t&#124; &lt; 2, so the
  null holds for both pairs. Descriptively, and explicitly **not** as a
  result (per the pre-registration, the total paired difference carries no
  directional claim): the *total* paired difference for ERC(S) &#8722;
  GMV(S) is +1.04%/yr full (t = 2.47, exceeding the &#124;t&#124; &lt; 2
  band) but +1.14%/yr test (t = 1.32, inside it), while ERC(S) &#8722;
  MDP(S)'s total is +0.40%/yr full (t = 0.99) and +0.04%/yr test
  (t = 0.06). The &#946; decomposition shows the ERC&#8722;GMV total gap is
  almost entirely the beta part (&Delta;&beta; part = +0.95%/yr vs.
  &Delta;&alpha; = +0.09%/yr) &#8212; consistent with ERC's higher ex-ante
  vol on the same &Sigma;, not with anything about the ERC construction
  itself, which is exactly why H3 was registered on &Delta;&alpha; rather
  than on the total.
- **H4 — supported, borderline (same pattern as notebook 05):** ERC(S)
  &#8722; EW is &#8722;2.78%/yr full (t = &#8722;2.08, just past the
  &#8722;2 threshold) and the test-window mean is also negative
  (&#8722;3.16%/yr, though its own t is only &#8722;1.15).
  &#124;&Delta;&beta; part&#124; = 3.53%/yr &gt; &#124;&Delta;&alpha;&#124;
  = 0.76%/yr, and &Delta;&alpha;'s own t (direct diff-regression) = 0.89
  &lt; 2 &#8212; consistent with a beta-driven shortfall, as the hypothesis
  predicted, not an alpha effect. The full-window t sits only marginally
  past &#8722;2, so this reads as a marginal pass, not a decisive one.
- **H5 — both nulls hold, one narrowly:** ERC(LW) &#8722; ERC(S) is
  +0.18%/yr full (t = 1.75) and +0.24%/yr test (t = 1.24) &#8212; both
  &#124;t&#124; &lt; 2, but the full-window t sits close to the 2 threshold,
  so this null holds narrowly rather than cleanly (the same word notebook
  05 used for its own t = 1.98 null). ERC(S) &#8722; IV &Delta;&alpha;
  (direct diff-regression) is +0.33%/yr (t = 0.56), comfortably inside the
  null band. It is also worth disclosing why ERC(LW) was backtested as a
  sensitivity at all: the read-only design check found
  max half-L1(ERC(LW), ERC(S)) = 0.0307 over the three snapshot dates
  against a pre-set 3% threshold &#8212; it cleared that bar by about 7 bp,
  on one of the three dates. That is a weak basis for treating the LW
  sensitivity as a meaningfully different portfolio in the first place, so
  its narrow null here should be read as consistent with "there was barely
  a difference to test," not as a strong robustness result.
- **Cost drag is small everywhere** (0.01&#8211;0.07%/yr across all six
  runs) and does not change any of the above conclusions; ERC(S)'s own
  turnover (26.6%/yr) sits between EW's (13.6%/yr) and MDP(S)'s (52.5%/yr).
- **nb05 link:** notebook 05 found MDP keeps the UUP corner that survives
  covariance shrinkage; this notebook finds ERC shrinks that corner in
  relative terms (48% vs. 58&#8211;59%) but does not remove it, and the
  mechanism is legible for the first time via Identity A &#8212; a
  low-&beta;-to-portfolio asset needs a large cash weight to carry an equal
  share of risk. No claim beyond what is tested here is made: this
  notebook does not test whether the corner is "wrong," does not compare
  ERC against a risk-budgeted (non-equal) alternative, and does not
  attribute UUP's low &beta;<sub>i,p</sub> to any specific macro regime
  &#8212; those are Phase 2 questions.